In [ ]:

import json
import os
import sys
from dotenv import load_dotenv
from pathlib import Path

%load_ext autoreload
%autoreload 2


load_dotenv()
root_path = Path().absolute().parent.parent.parent
sys.path.append(str(root_path))


from functions.llm_config import LLMConfig
from functions.retrieval import QuestionRetriever
from functions.query_decomposer import QueryDecomposer
from functions.sqldatabase_langchain_utils import SQLDatabaseLangchainUtils
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage
import paths as paths

import re
from pydantic import BaseModel, Field
from typing import List



from eval_agent.text2sql_agent.text_to_sql.text_to_sql_extended_schema import TextToSQLExtendedSchema




The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## MONDIAL GPT

In [24]:

O3 = LLMConfig(provider="azure").get_llm(model="o3-mini", reasoning_effort="high")
GPT4O = LLMConfig(provider="azure").get_llm(model="gpt-4o")



decomposer = QueryDecomposer(
    GPT4O,
    paths.PROMPT_DECOMPOSER_FILE,
    False
)



experiment = "mondial"

if "mondial" in experiment:
    prompt_path = paths.MONDIAL_GPT_EXTENDED_SCHEMA_PROMPT
    DATASET_SYNTHETIC_PATH = paths.DATASET_SYNTHETIC_PATH
    EMBEDDINGS_PATH = paths.EMBEDDINGS_PATH

else:
    raise ValueError("Invalid experiment name. Use 'mondial' .")


with open(paths.DB_CONNECTION_FILE, "r") as f:
    db_connection = json.load(f)

retriever = QuestionRetriever(
    dataset_path=DATASET_SYNTHETIC_PATH,
    vectors_path=EMBEDDINGS_PATH,
    # vectorize=True
)
retriever.remove_duplicates()

17030
17030


In [ ]:
sqldatabase = SQLDatabaseLangchainUtils(db_connection)

In [26]:
t =  TextToSQLExtendedSchema(GPT4O, decomposer, retriever, prompt_path, debug=False)


def test(question):
    print("Running Test")

    result = t.translate_text_to_sql(question)

    
    result.sql_query = result.sql_query.replace("```sql", "").replace("```", "")
    if not result.sql_query.strip().upper().startswith("SELECT"):
        result.sql_query = "SELECT " + result.sql_query
    if "DISTINCT" in result.sql_query:
        result.sql_query = result.sql_query.replace("DISTINCT", "")
    print("Result")
    sql = result.sql_query
    print(result.sql_query)
    print(result.schema_linking_tables)
    print(sqldatabase.run_in_database(sql))
    

In [27]:
test("capital of cuba")

Running Test
Decomposition
['capital of cuba', 'Qual é a capital de Cuba?']
(17030,)
(17030,)
DFE
Question: islands name 'Cuba'
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: islands name "Cuba"
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: cities on island Lesser Antilles
SELECT MONDIAL_CITY.NAME
FROM MONDIAL_CITY
JOIN MONDIAL_AIRPORT ON MONDIAL_CITY.NAME = MONDIAL_AIRPORT.CITY
JOIN MONDIAL_ISLAND ON MONDIAL_AIRPORT.ISLAND = MONDIAL_ISLAND.NAME
WHERE LOWER(MONDIAL_ISLAND.NAME) = 'lesser antilles'

Question: regions specific city designated as capital
SELECT NAME FROM MONDIAL_PROVINCE WHERE CAPPROV IS NOT NULL;

Question: countries with 'Madrid' as capital
SELECT NAME FROM MONDIAL_COUNTRY WHERE LOWER(CAPITAL) = 'madrid'

Question: islands name 'Cuba'
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: islands name "Cuba"
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: Which islands in the database ha

In [ ]:
from eval_agent.text2sql_agent.tool_mondial import convert_text_to_sql_and_execute

print(convert_text_to_sql_and_execute("Qual é a capital de cuba?"))

Decomposition
['Qual é a capital de cuba?', 'Qual é a capital de Cuba?']
(17030,)
(17030,)
DFE
Question: islands name 'Cuba'
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: islands name "Cuba"
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: What are the islands in the MONDIAL_ISLAND table with the name "Cuba"?
SELECT NAME, ISLANDS, AREA, ELEVATION, TYPE, COORDINATES, META_REPCOL
FROM MONDIAL_ISLAND
WHERE LOWER(NAME) = 'cuba'

Question: Which islands in the database have the name 'Cuba'?
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: Which islands in the database have the name "Cuba"?
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: islands name 'Cuba'
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: islands name "Cuba"
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: Which islands in the database have the name "Cuba"?
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: What are the islands in the MONDIAL_ISLAND table with the name "Cuba"?
SELECT NAME, ISLANDS, AREA, ELEVATION, TYPE, COORDINATES, META_REPCOL
FROM MONDIAL_ISLAND
WHERE LOWER(NAME) = 'cuba'

Question: Which islands in the database have the name 'Cuba'?
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'


{'input': 'Qual é a capital de cuba?', 'schema_linking': ['mondial_gpt.country'], 'answer':      CAPITAL
0  La Habana, 'sql': "SELECT capital FROM mondial_gpt.country WHERE LOWER(name) = 'cuba'"}
